In [4]:
# run_delong_comparison.py
import os
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
# 确保 MLstatkit 已安装: pip install MLstatkit
from MLstatkit.stats import Delong_test # 检查导入路径是否正确，有时可能是 from mlstatkit.stats import Delong_test

# --- 基本路径配置 ---
BASE_DIR_TRADITIONAL_ML = "./traditional_ml_outputs" # 你传统模型输出的根目录
BASE_DIR_DEEP_LEARNING = "./dl_model_outputs_replication" # 你深度学习模型输出的根目录
OUTPUT_CSV_DIR = "./delong_test_results" # 保存DeLong检验结果的目录
os.makedirs(OUTPUT_CSV_DIR, exist_ok=True)

# --- 数据集和模型配置 ---
# 数据集标签 (用于构建路径和报告)
DATASET_TAGS = {
    "mimic3_36": "mimic3_36_factors",
    "mimic4_36": "mimic4_36_factors",
    "mimic3_eICU_8": "mimic3_eICU_8_factors",
    "mimic4_8": "mimic4_8_factors",
    "local_8": "local_8_factors"
}

# 模型名称映射 (键: 你在报告中想用的名字, 值: 文件夹名称中对应的部分或完整标识)
# 你需要仔细检查 dl_model_outputs_replication 目录下的文件夹命名规则来确定这些
# 对于传统模型，我们用的是 "DT_", "LR_", "RF_", "SGD_", "SVM_" 前缀
# 对于深度学习模型，我们用的是模型名如 "Transformer_", "Lstm_", "GRU_", "GPT_"
MODELS_CONFIG = {
    "SepsisFormer (Transformer)": {"type": "dl", "dir_prefix": "Transformer_", "label_file": "test_labels.npy", "prob_file": "test_probabilities.npy"},
    "LSTM": {"type": "dl", "dir_prefix": "Lstm_", "label_file": "test_labels.npy", "prob_file": "test_probabilities.npy"},
    "GRU": {"type": "dl", "dir_prefix": "GRU_", "label_file": "test_labels.npy", "prob_file": "test_probabilities.npy"},
    "GPT": {"type": "dl", "dir_prefix": "GPT_", "label_file": "test_labels.npy", "prob_file": "test_probabilities.npy"},
    "Decision Tree": {"type": "ml", "dir_prefix": "DT_", "label_file": "test_labels.npy", "prob_file": "test_probabilities.npy"},
    "Logistic Regression": {"type": "ml", "dir_prefix": "LR_", "label_file": "test_labels.npy", "prob_file": "test_probabilities.npy"},
    "Random Forest": {"type": "ml", "dir_prefix": "RF_", "label_file": "test_labels.npy", "prob_file": "test_probabilities.npy"},
    "SGD Classifier": {"type": "ml", "dir_prefix": "SGD_", "label_file": "test_labels.npy", "prob_file": "test_probabilities.npy"},
    "SVM": {"type": "ml", "dir_prefix": "SVM_", "label_file": "test_labels.npy", "prob_file": "test_probabilities.npy"},
}

# --- 辅助函数 ---
def find_latest_run_dir(base_path, model_dir_prefix, dataset_tag_in_dirname):
    """
    查找包含特定标识的最新运行目录。
    注意：这依赖于你的目录命名中时间戳是最后一部分且按时间排序。
    对于深度学习模型，dataset_tag_in_dirname 可能是 dataset_tag 本身或加上额外标识。
    """
    candidate_dirs = []
    if not os.path.exists(base_path):
        print(f"警告: 基础路径不存在: {base_path}")
        return None
        
    for dirname in os.listdir(base_path):
        if dirname.startswith(model_dir_prefix) and dataset_tag_in_dirname in dirname:
            candidate_dirs.append(dirname)
    
    if not candidate_dirs:
        return None
    
    # 按名称排序（通常时间戳在后面，所以能取到最新的）
    candidate_dirs.sort()
    return os.path.join(base_path, candidate_dirs[-1])


def load_predictions(directory_path, label_filename="test_labels.npy", prob_filename="test_probabilities.npy"):
    """从 .npy 文件加载真实标签和预测概率。"""
    if directory_path is None or not os.path.isdir(directory_path):
        # print(f"  - 警告: 目录不存在或为None: {directory_path}")
        return None, None

    labels_path = os.path.join(directory_path, label_filename)
    probs_path = os.path.join(directory_path, prob_filename)

    if not os.path.exists(labels_path):
        print(f"  - 警告: 标签文件未找到: {labels_path}")
        return None, None
    if not os.path.exists(probs_path):
        print(f"  - 警告: 概率文件未找到: {probs_path}")
        return None, None
            
    try:
        y_true = np.load(labels_path)
        y_probs = np.load(probs_path)
        if y_true is None or y_probs is None: # 进一步检查加载的内容
            print(f"  - 警告: 从 {directory_path} 加载的标签或概率为None。")
            return None, None
        if len(y_true) == 0 or len(y_probs) == 0:
            print(f"  - 警告: 从 {directory_path} 加载的标签或概率为空数组。")
            return None, None
        if len(y_true) != len(y_probs):
            print(f"  - 警告: 从 {directory_path} 加载的标签和概率长度不匹配: {len(y_true)} vs {len(y_probs)}。")
            return None, None
        return y_true, y_probs
    except Exception as e:
        print(f"  - 错误: 从 {directory_path} 加载预测结果时出错: {e}")
        return None, None

# --- 主逻辑 ---
all_results_summary = []

for report_tag, dataset_actual_tag_part in DATASET_TAGS.items():
    print(f"\n\n{'='*20} 开始处理数据集: {report_tag} (基于标识: {dataset_actual_tag_part}) {'='*20}")
    
    dataset_predictions = {}
    y_true_reference = None
    reference_model_name = "SepsisFormer (Transformer)" # 基准模型

    # 1. 加载所有模型的预测结果
    print(f"--- 正在为数据集 '{report_tag}' 加载预测结果 ---")
    for model_report_name, config in MODELS_CONFIG.items():
        print(f"  正在加载模型: {model_report_name}")
        model_dir_path = None
        if config["type"] == "ml":
            # 传统模型的目录名通常是: ML_MODEL_PREFIX_dataset_tag_notebook
            # 例如: DT_mimic3_36_factors_notebook
            model_dir_name = f"{config['dir_prefix']}{dataset_actual_tag_part}_notebook"
            model_dir_path = os.path.join(BASE_DIR_TRADITIONAL_ML, model_dir_name)
        elif config["type"] == "dl":
            # 深度学习模型的目录名更复杂，包含时间戳，需要查找
            # 我们需要一个在目录名中能匹配 dataset_actual_tag_part 的部分
            # 例如，对于 Transformer_mimic3_36_factors_mimic3-36f-transformer-replication_trainseed42_...
            # dataset_tag_in_dirname 可能是 dataset_actual_tag_part (如 "mimic3_36_factors")
            # 或者更具体的，例如 "mimic3_36_factors_mimic3-36f-transformer-replication"
            # 为了简单，我们先假设 dataset_actual_tag_part 存在于目录名中
            # 对于微调模型，它们的文件名中可能包含 "finetune" 和目标数据集的tag (如 mimic4_8_factors)
            # 而预训练模型名中包含源数据集的tag (如 mimic3_eICU_8_factors)

            # 这是一个启发式方法，你可能需要根据DL模型的实际目录命名规则精确调整 dataset_tag_in_dirname
            # 例如，如果DL目录总是 "MODELPREFIX_DATASETTAG_..."
            # dataset_tag_in_dirname_for_dl = dataset_actual_tag_part.replace("_split","") # 从 presplit_data_dir 获取的tag
            # 假设你的 DL 目录名中，dataset_actual_tag_part 是一个可靠的子串
            # 对于微调模型，它们文件名中会有目标数据集的tag，所以用 dataset_actual_tag_part
            # 对于从头训练的DL模型，文件名中也会有目标数据集的tag
            
            # 检查你的dl_model_outputs_replication文件夹结构
            # 假设文件夹名类似于：MODEL_dataset_tag_SOME_OTHER_INFO_TIMESTAMP
            # 例如：Transformer_mimic3_36_factors_mimic3-36f-transformer-replication_trainseed42_...
            # 我们需要一个能在目录名中唯一识别出对应数据集运行的部分
            # 对于 "mimic3_36", 文件夹名中的 dataset_tag_in_dirname 应该是 "mimic3_36_factors"
            
            # 简化查找：我们假设目录名中明确包含了 dataset_actual_tag_part
            model_dir_path = find_latest_run_dir(BASE_DIR_DEEP_LEARNING, config["dir_prefix"], dataset_actual_tag_part)
            if model_dir_path is None:
                 # 尝试另一种可能的命名 (例如，如果finetune的tag不同)
                 # 这部分逻辑可能需要根据你的确切命名规则来完善
                 if "finetune" in model_report_name.lower() or "finetune" in config.get("tag_hint","").lower(): # 假设config可以有个tag_hint
                     pass # 这里可以添加更复杂的查找逻辑

        if model_dir_path is None:
            print(f"    - 未找到模型 {model_report_name} 在数据集 {report_tag} 上的运行目录。跳过。")
            continue
        if not os.path.isdir(model_dir_path):
            print(f"    - 目录不存在: {model_dir_path} for model {model_report_name} on dataset {report_tag}。跳过。")
            continue

        print(f"    - 使用目录: {model_dir_path}")
        y_true, y_probs = load_predictions(model_dir_path, config["label_file"], config["prob_file"])

        if y_true is not None and y_probs is not None:
            if y_true_reference is None and model_report_name == reference_model_name:
                y_true_reference = y_true.astype(int) # 确保是整数类型
                print(f"    - 参考真实标签已从 {reference_model_name} 设置 (长度: {len(y_true_reference)})。")
            
            dataset_predictions[model_report_name] = {"labels": y_true.astype(int), "probs": y_probs}
        else:
            print(f"    - 未能加载 {model_report_name} 在数据集 {report_tag} 上的预测结果。")

    if y_true_reference is None and reference_model_name in dataset_predictions:
        # 如果SepsisFormer不是第一个成功加载的，但已加载，则用它的标签作参考
        y_true_reference = dataset_predictions[reference_model_name]["labels"]
        print(f"    - 参考真实标签已从 {reference_model_name} 设置 (长度: {len(y_true_reference)}) (非首次加载)。")
    elif y_true_reference is None:
        print(f"错误: 未能加载基准模型 {reference_model_name} 的预测结果 for dataset {report_tag}。无法进行比较。")
        continue # 跳到下一个数据集

    # 2. 验证真实标签一致性并计算AUC
    print(f"\n--- 为数据集 '{report_tag}' 计算AUC并验证标签 ---")
    auc_scores = {}
    valid_predictions_for_delong = {}

    for model_name, data in dataset_predictions.items():
        if not np.array_equal(y_true_reference, data["labels"]):
            print(f"  !!! 严重错误: 模型 '{model_name}' 的真实标签与参考标签不一致 for dataset {report_tag}。跳过此模型。")
            print(f"    参考标签前10: {y_true_reference[:10]}")
            print(f"    当前标签前10: {data['labels'][:10]}")
            continue
        
        if len(np.unique(data["labels"])) < 2:
            auc = np.nan # 或者0.0，但nan更能表示无法计算
            print(f"  模型 '{model_name}': 测试集只包含一个类别，无法计算AUC。")
        elif data["probs"] is None or len(data["probs"]) != len(data["labels"]):
            auc = np.nan
            print(f"  模型 '{model_name}': 预测概率缺失或长度不匹配。")
        else:
            try:
                auc = roc_auc_score(data["labels"], data["probs"])
            except ValueError as e:
                print(f"  模型 '{model_name}': 计算AUC时出错: {e} (可能概率值有问题，例如都是常数)。")
                auc = np.nan

        auc_scores[model_name] = auc
        print(f"  模型 '{model_name}': AUC = {auc:.4f}" if not np.isnan(auc) else f"  模型 '{model_name}': AUC = N/A")
        if not np.isnan(auc): # 只把能计算AUC的模型用于DeLong检验
             valid_predictions_for_delong[model_name] = data["probs"]


    # 3. 执行DeLong检验 (SepsisFormer vs. 其他模型)
    print(f"\n--- 为数据集 '{report_tag}' 执行DeLong检验 (基准: {reference_model_name}) ---")
    if reference_model_name not in valid_predictions_for_delong:
        print(f"  错误: 基准模型 {reference_model_name} 没有有效的预测概率。无法进行DeLong检验。")
        continue

    sepsisformer_probs = valid_predictions_for_delong[reference_model_name]
    results_delong_dataset = []

    for model_name, other_model_probs in valid_predictions_for_delong.items():
        if model_name == reference_model_name:
            continue
        
        try:
            # 使用位置参数调用 Delong_test
            z_score, p_value = Delong_test(y_true_reference, sepsisformer_probs, other_model_probs) # <<--- 修改后的调用
            
            results_delong_dataset.append({
                "Dataset": report_tag,
                "Model A (Baseline)": reference_model_name,
                "Model B": model_name,
                "AUC A": auc_scores.get(reference_model_name, np.nan),
                "AUC B": auc_scores.get(model_name, np.nan),
                "Z-Score (A vs B)": z_score,
                "P-Value (A vs B)": p_value,
                # 保留或调整显著性判断逻辑
                "A significantly better than B (p<0.05)": "Yes" if p_value < 0.05 and auc_scores.get(reference_model_name, 0) > auc_scores.get(model_name, 0) else "No",
                "B significantly better than A (p<0.05)": "Yes" if p_value < 0.05 and auc_scores.get(model_name, 0) > auc_scores.get(reference_model_name, 0) else "No",
                "No significant difference (p>=0.05)": "Yes" if p_value >=0.05 else "No" # 新增一个判断
            })
            print(f"  {reference_model_name} (AUC: {auc_scores.get(reference_model_name, np.nan):.4f}) vs. {model_name} (AUC: {auc_scores.get(model_name, np.nan):.4f}): Z={z_score:.4f}, P={p_value:.4f}")
        except Exception as e:
            print(f"  执行DeLong检验时出错 ({reference_model_name} vs. {model_name}): {e}")
            results_delong_dataset.append({
                "Dataset": report_tag,
                "Model A (Baseline)": reference_model_name,
                "Model B": model_name,
                "AUC A": auc_scores.get(reference_model_name, np.nan),
                "AUC B": auc_scores.get(model_name, np.nan),
                "Z-Score (A vs B)": np.nan,
                "P-Value (A vs B)": np.nan,
                "Error": str(e) # 将错误信息也记录下来
            })
            
    if results_delong_dataset:
        df_delong_dataset = pd.DataFrame(results_delong_dataset)
        csv_filename = os.path.join(OUTPUT_CSV_DIR, f"delong_results_{report_tag.replace(' ', '_')}.csv")
        df_delong_dataset.to_csv(csv_filename, index=False)
        print(f"  DeLong检验结果已保存至: {csv_filename}")
        all_results_summary.extend(results_delong_dataset) # 添加到总摘要
    else:
        print(f"  未能为数据集 {report_tag} 执行任何DeLong检验。")

# 保存所有数据集的DeLong检验总摘要
if all_results_summary:
    df_all_summary = pd.DataFrame(all_results_summary)
    summary_csv_filename = os.path.join(OUTPUT_CSV_DIR, "delong_results_ALL_DATASETS_summary.csv")
    df_all_summary.to_csv(summary_csv_filename, index=False)
    print(f"\n\n所有数据集的DeLong检验总摘要已保存至: {summary_csv_filename}")
else:
    print("\n\n未能生成任何DeLong检验结果。")

print("\nDeLong检验脚本执行完毕。")



==================== 开始处理数据集: mimic3_36 (基于标识: mimic3_36_factors) ====================
--- 正在为数据集 'mimic3_36' 加载预测结果 ---
  正在加载模型: SepsisFormer (Transformer)
    - 使用目录: ./dl_model_outputs_replication/Transformer_mimic3_36_factors_mimic3-36f-transformer-replication_trainseed42_20250610-031107
    - 参考真实标签已从 SepsisFormer (Transformer) 设置 (长度: 1144)。
  正在加载模型: LSTM
    - 使用目录: ./dl_model_outputs_replication/Lstm_mimic3_36_factors_mimic3-36f-lstm-replication_trainseed42_20250610-032242
  正在加载模型: GRU
    - 使用目录: ./dl_model_outputs_replication/GRU_mimic3_36_factors_mimic3-36f-gru-replication_trainseed42_20250610-032613
  正在加载模型: GPT
    - 使用目录: ./dl_model_outputs_replication/GPT_mimic3_36_factors_mimic3-36f-gpt-replication_trainseed42_20250610-034126
  正在加载模型: Decision Tree
    - 使用目录: ./traditional_ml_outputs/DT_mimic3_36_factors_notebook
  正在加载模型: Logistic Regression
    - 使用目录: ./traditional_ml_outputs/LR_mimic3_36_factors_notebook
  正在加载模型: Random Forest
    - 使用目录: ./traditional_ml_o